# Extensions of the project

In [ ]:
import sys
import os
import subprocess

print("Initialisation de l'environnement...")

# --- 1. GESTION DES LIBRAIRIES EXTERNES (Téléchargement auto) ---
def install_package(package):
    print(f"Téléchargement de '{package}' en cours...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

try:
    # On essaie d'importer les librairies "publiques"
    import torchvision
    import torch
    import pandas as pd
    import time
    import matplotlib.pyplot as plt
    from tqdm import tqdm
    import seaborn as sns
    print("Toutes les librairies externes sont déjà présentes.")

except ImportError as e:
    # Si une librairie manque (ex: seaborn), on installe tout le paquet d'un coup
    print(f"Il manque des librairies (Erreur: {e}). Installation en cours...")
    
    # On installe les classiques qui manquent souvent sur SSP Cloud
    pkgs = ["pandas", "matplotlib", "seaborn", "tqdm", "torch", "torchvision"]
    for pkg in pkgs:
        try:
            __import__(pkg) # Tente un import basique
        except ImportError:
            install_package(pkg)
            
    # On refait les imports après installation
    import torchvision
    import torch
    import pandas as pd
    import time
    import matplotlib.pyplot as plt
    from tqdm import tqdm
    import seaborn as sns
    print("Installation terminée et librairies chargées.")


# --- 2. GESTION DE TES FICHIERS LOCAUX (Neural_networks.py, etc) ---
try:
    # C'est ici qu'on charge ton code à toi
    from Neural_networks import *
    from Comparing_results import *
    from Extension_analysis_functions import *
    print("Tes fichiers locaux (Neural_networks, Comparing_results, Extension_analysis_functions) sont bien chargés.")

except ImportError as e:
    print("\n" + "_"*60)
    print(f"ERREUR : Impossible de trouver le fichier '{e.name}'.")
    print("SOLUTION : Tu dois 'Uploader' (Glisser-Déposer) tes fichiers .py")
    print("(Neural_networks.py, Comparing_results.py, Helper_functions.py)")
    print("dans la barre de gauche de VS Code ou Jupyter.")
    print("_"*60 + "\n")


# --- 3. VÉRIFICATION GPU (Pour être sûr que tu vas vite) ---
if torch.cuda.is_available():
    print(f"\nEXCELLENT : GPU détecté -> {torch.cuda.get_device_name(0)}")
else:
    print("\nATTENTION : Aucun GPU détecté. L'entraînement sera lent (CPU).")

## Extension: "Late Rewinding" (Reset to Epoch k)

This extension is applied when standard Lottery Ticket Hypothesis experiments fail to identify winning tickets, typically on complex datasets like CIFAR-10 or with deeper network architectures. Resetting weights strictly to initialization (Epoch 0) can be unstable because the optimization trajectory is highly sensitive to stochastic noise during the very first iterations. Late Rewinding addresses this by resetting the pruned network's weights to their state after a small number of warmup epochs (e.g., Epoch 2) instead of the initial state. By this point, the network has entered a stable optimization basin, allowing the sparse subnetwork to train successfully without diverging. Consequently, this method maintains high accuracy at extreme sparsity levels where the standard initialization approach often collapses.

In [ ]:
# --- PARAMÈTRES DE L'EXTENSION ---
# On pousse le pruning très loin (98%) pour voir la différence de stabilité
TOTAL_PRUNE_PERCENT = 98
ROUNDS = 10
EPOCHS = 15
REPEATS = 3
REWIND_EPOCH = 2 # On reset à l'époque 2 (Standard LTH = Reset à 0)

# --- DÉFINITION DES MÉTHODES ---

# Méthode 1 : Standard LTH (Reset à 0)
def method_standard_lth():
    return iterative_pruning_CIFAR(
        total_prune_percent=TOTAL_PRUNE_PERCENT,
        rounds=ROUNDS,
        epochs_per_round=EPOCHS,
        lr=0.1,
        LTH=True,
        optimizer_type="sgd"
    )

# Méthode 2 : Late Rewinding (Reset à l'époque 2)
def method_late_rewinding():
    return iterative_pruning_CIFAR_Rewinding(
        total_prune_percent=TOTAL_PRUNE_PERCENT,
        rounds=ROUNDS,
        epochs_per_round=EPOCHS,
        rewind_epoch=REWIND_EPOCH, # <-- Le changement clé
        lr=0.1,
        optimizer_type="sgd"
    )

print("🚀 Lancement de la comparaison : Standard vs Late Rewinding sur CIFAR-10...")

df_std, df_rewind = comparing_methods_initialization_after_pruning(
    amount_of_repeats=REPEATS,
    rounds=ROUNDS,
    method_1=method_standard_lth,
    method_2=method_late_rewinding
)

# --- AFFICHAGE ---
comparing_methods_plotting(
    df_std, 
    df_rewind, 
    "Standard LTH (Reset Epoch 0)", 
    f"Late Rewinding (Reset Epoch {REWIND_EPOCH})"
)

## Extension: "Strong Lottery Ticket Hypothesis" (Pruning without Training)

Based on the theoretical proofs by Da Cunha et al. (2022), the Strong Lottery Ticket Hypothesis asserts that sufficiently overparameterized networks contain subnetworks that perform well without any weight training. This extension applies when investigating the intrinsic representational power of random initializations, where the goal is to optimize only the connectivity (the binary mask) while keeping the random weights frozen. Unlike the standard LTH which requires expensive retraining steps, this method demonstrates that finding the correct topological structure within random noise is sufficient for high performance. This effectively shifts the focus from optimizing parameter values to optimizing network architecture (pruning) as the primary learning mechanism.

In [ ]:
# Test rapide : Est-ce qu'on peut atteindre une bonne perf en supprimant 50% des poids
# sans JAMAIS entraîner les valeurs des poids ?
acc = run_strong_lth_experiment(target_prune_percent=50, epochs=10, lr=0.1)

# Pour ton rapport, tu peux faire une boucle :
# sparsities = [10, 30, 50, 70, 90, 95]
# accuracies = []
# for s in sparsities:
#     acc = run_strong_lth_experiment(target_prune_percent=s, epochs=20)
#     accuracies.append(acc)
# Puis tracer la courbe !

In [ ]:


# --- 1. PARAMÈTRES DE L'EXPÉRIENCE ---
# Liste des sparsités à tester.
# On insiste sur la fin (90-99%) car c'est là que c'est impressionnant.
target_sparsities = [0, 20, 50, 70, 90, 95, 98, 99] 
EPOCHS = 20  # Assez rapide, car on entraîne juste les scores (masques)
LR = 0.1     # Learning Rate pour les scores (Edge-Popup)

results = []

print("🚀 Lancement du 'Sweep' Strong LTH (Analyse de la sparsité)...")

# --- 2. BOUCLE D'EXPÉRIENCE ---
for sparsity in tqdm(target_sparsities, desc="Testing Sparsities"):
    # On lance l'expérience pour ce niveau de sparsité
    acc = run_strong_lth_experiment(
        target_prune_percent=sparsity, 
        epochs=EPOCHS, 
        lr=LR
    )
    
    results.append({
        "Sparsity (%)": sparsity,
        "Test Accuracy (%)": acc
    })

# Création d'un DataFrame pour voir les chiffres
df_results = pd.DataFrame(results)
print("\n📊 Résultats bruts :")
print(df_results)

# --- 3. PLOTTING (GRAPHIQUE RAPPORT) ---
plt.figure(figsize=(10, 6))

# Courbe Strong LTH
plt.plot(df_results["Sparsity (%)"], df_results["Test Accuracy (%)"], 
         marker='o', linestyle='-', linewidth=2, markersize=8, color='purple', label="Strong LTH (Random Weights + Learned Mask)")

# Ligne de base : Performance aléatoire (10% sur CIFAR-10)
plt.axhline(y=10, color='gray', linestyle='--', alpha=0.5, label="Random Guessing (10%)")

# (Optionnel) Ligne de référence : Performance d'un réseau entraîné normalement (~60-65% pour Conv2)
# Vous pouvez décommenter cette ligne pour montrer la différence avec un entraînement classique
# plt.axhline(y=65, color='green', linestyle='--', alpha=0.5, label="Standard Training Baseline")

# Esthétique
plt.title("Strong Lottery Ticket Hypothesis: Performance without Weight Training", fontsize=14)
plt.xlabel("Sparsity (Percentage of Weights Removed)", fontsize=12)
plt.ylabel("Test Accuracy (%)", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.ylim(0, 100) # Pour bien voir l'échelle globale

# Sauvegarde pour le rapport
plt.tight_layout()
plt.savefig("Strong_LTH_Results.png", dpi=300)
plt.show()